In [6]:
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D, TextVectorization

2026-02-22 16:17:18.350618: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-22 16:17:18.644729: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-22 16:17:26.494502: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [45]:
dataset = pd.read_csv('transactions_dataset.csv')
dataset.head()

,Text,Category Id
0,Lidl )))),2
1,Deposit Rent,1
2,McDonalds Banegaards )))),3
3,Burger Shack,3
4,Netto,2


In [34]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/mukama/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [35]:
# Splitting the dataset into the Training set and Test set
from sklearn.model_selection import train_test_split
X = dataset.iloc[:, 0]
y = dataset.iloc[:, 1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

In [36]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1200,), (300,), (1200,), (300,))

In [37]:
# Create a custom standardization function
def custom_standardization(input_data):
  text = tf.strings.lower(input_data)
  text = tf.strings.regex_replace(text, '[^a-zA-Z0-9]', ' ')
  # text = [ps.stem(word) for word in text if not word in set(stopwords.words('english'))]
  # text = [ps.stem(word) for word in text if not word in set(stopwords.words('danish'))]
  return text

# Vocabulary size and number of words in a sequence.
vocab_size = 200
sequence_length = 30

# Use the text vectorization layer to normalize, split, and map strings to 
# integers. Note that the layer uses the custom standardization defined above. 
# Set maximum_sequence length as all samples are not of the same length.
vectorize_layer = TextVectorization(
    standardize=custom_standardization,
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=sequence_length)

# Make a text-only dataset (no labels) and call adapt to build the vocabulary.
vectorize_layer.adapt(X_train.to_list())

In [38]:
embedding_dim=5

model = tf.keras.models.Sequential([
  vectorize_layer,
  Embedding(vocab_size, embedding_dim, name="embedding"), # Embed a 200 word vocabulary into 5 dimensions
  GlobalAveragePooling1D(),
  Dense(16, activation='relu'),
  Dense(10)
])

In [39]:
# Pass as a single tensor so Keras 3 accepts it as one positional input
predictions = model(tf.constant(X_train.to_list()[:3])).numpy()
predictions

array([[ 0.0157769 , -0.0090353 ,  0.03540839,  0.00086114,  0.03280665,
        -0.00258986,  0.02440304,  0.03533687, -0.00548226, -0.00427977],
       [ 0.01734147, -0.01099042,  0.03543941, -0.00142703,  0.03433105,
        -0.00157287,  0.0258501 ,  0.03783042, -0.00638363, -0.00379257],
       [ 0.01674673, -0.01070108,  0.03583571, -0.00080648,  0.03393365,
        -0.00118158,  0.02632848,  0.03646157, -0.00690464, -0.00481125]],
      dtype=float32)

In [40]:
tf.nn.softmax(predictions).numpy()

array([[0.10033095, 0.09787215, 0.10232005, 0.09884554, 0.10205419,
        0.09850501, 0.10120016, 0.10231274, 0.09822051, 0.09833869],
       [0.10045183, 0.09764577, 0.10228635, 0.09858408, 0.10217305,
        0.0985697 , 0.10131019, 0.10253121, 0.09809665, 0.09835115],
       [0.10040966, 0.09769112, 0.10234479, 0.09866253, 0.10215031,
        0.09862553, 0.10137638, 0.10240886, 0.0980627 , 0.0982682 ]],
      dtype=float32)

In [41]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

In [42]:
loss_fn(tf.constant(y_train.to_list()[:3]), predictions).numpy()

2026-02-22 16:52:55.356707: W tensorflow/core/framework/op_kernel.cc:1831] OP_REQUIRES failed at cast_op.cc:122 : UNIMPLEMENTED: Cast string to float is not supported
2026-02-22 16:52:55.356754: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: UNIMPLEMENTED: Cast string to float is not supported


UnimplementedError: {{function_node __wrapped__Cast_device_/job:localhost/replica:0/task:0/device:CPU:0}} Cast string to float is not supported [Op:Cast] name: 

In [43]:
model.compile(optimizer='adam',
              loss=loss_fn,
              metrics=['accuracy'])

In [44]:
# Use Dataset so Keras 3 doesn't hit 'Invalid dtype: str' with string tensors in tree
train_ds = tf.data.Dataset.from_tensor_slices((tf.constant(X_train.to_list()), tf.constant(y_train.to_list()))).batch(32)
model.fit(train_ds, epochs=250)

Epoch 1/250


2026-02-22 16:53:06.628151: W tensorflow/core/framework/op_kernel.cc:1831] OP_REQUIRES failed at cast_op.cc:122 : UNIMPLEMENTED: Cast string to float is not supported


UnimplementedError: Graph execution error:

Detected at node compile_loss/sparse_categorical_crossentropy/Cast defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/home/mukama/Documents/transaction-categorization-main/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/mukama/Documents/transaction-categorization-main/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/mukama/Documents/transaction-categorization-main/.venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start

  File "/home/mukama/Documents/transaction-categorization-main/.venv/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/usr/lib/python3.12/asyncio/base_events.py", line 641, in run_forever

  File "/usr/lib/python3.12/asyncio/base_events.py", line 1987, in _r

In [26]:
# Use Dataset to avoid Keras 3 'Invalid dtype: str' with string inputs
test_ds = tf.data.Dataset.from_tensor_slices((tf.constant(X_test.to_list()), tf.constant(y_test.to_list()))).batch(32)
model.evaluate(test_ds, verbose=2)

2/2 - 0s - 80ms/step - accuracy: 0.3636 - loss: 1.7927


[1.7927491664886475, 0.3636363744735718]

In [32]:
categories = {
    0:'Automobile and Transport',
    1:'Housing and Real-Estate',
    2:'Groceries',
    3:'Recreation and Leisure',
    4:'Health and Well Being',
    5:'Hobby and Knowledge',
    6:'Clothes and Equipment',
    7:'Cash and Credit',
    8:'Financial Services',
    9:'Other'
}

def get_category_by_id(id):
    return categories[id];

inputs = ['lidl', 'netto ))', 'udemy', 'kfc', 'rent']
# predict_classes removed in Keras 3: use predict + argmax (model outputs logits)
pred_logits = model.predict(tf.constant(inputs))
predictions = tf.argmax(pred_logits, axis=-1).numpy()
{ inputs[id]: get_category_by_id(predictions[id]) for id in range(predictions.size) }

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step


{'lidl': 'Recreation and Leisure',
 'netto ))': 'Recreation and Leisure',
 'udemy': 'Hobby and Knowledge',
 'kfc': 'Recreation and Leisure',
 'rent': 'Housing and Real-Estate'}